# Environments, Reproducibilty, Debugging, and Speed Up
----

By Adam A Miller (Northwestern/CIERA/SkAI)  
17 Sept 2026

**Version 0.1**

Sometimes you need to protect yourself from ...

yourself

Today we investigate a number of practices that will make your life as a graduate student better as you work for a sustained period of several years. 

## Learning Objectives
1. **Environments** — keep your packages from fighting each other
2. **Reproducibility** — what that word actually means, precisely
3. **Debugging** — read the error message instead of guessing
4. **Profiling** — find out where the time actually goes

## Problem 1) Environments

**Why at the start of Onboard did I suggest you create a new `conda` environment to handle your work this week?**

Your `astropy` needs `numpy 1.24`.

Your `scikit-learn` needs `numpy 2.0`, it has hugely updated syntax that breaks everything in your `astropy`.

What is a computer supposed to do?!

Environments fix this problem. 

Each environment is a private, disposable sandbox that has:

- its own Python interpreter

- its own installed packages

- its own version numbers

Go ahead and break it beyond repair. 

It's all good - delete, rinse, wash, and repeat. Nothin else changes after the deletion.

There are two especially common environments for `python`

| | `venv` | `conda` |
|---|---|---|
| Ships with | Python itself | Anaconda / Miniconda |
| Manages | Python packages | Python and non-Python dependencies |
| Good for | Lightweight, pure-Python projects | Scientific stacks with compiled dependencies |

I recommend `conda`, mainly because so much scientific software depends on compiled libraries (BLAS, HDF5, MPI) that `pip` alone can't manage.

Creating environments is fairly simple

```conda create -n onboard python=3.11```  
`conda activate onboard`  
```conda install numpy pandas matplotlib```

In [ ]:
# venv (built into Python, no conda required)
python -m venv onboard-env
source onboard-env/bin/activate   # myproject-env\Scripts\activate on Windows
pip install numpy pandas matplotlib

I bet some (most? all?) of you create environments, but do you save them? 

This is easy, and remember today is about *saving yourself* 

exporting your environments allows you to rebuild them if your laptop dies.

(or you have a tendancy to trip while holding expensive equipment, like Adam...)

conda:  
`conda env export > environment.yml`

pip:  
`pip freeze > requirements.txt`

These environment files should sit in your repos. It allows anyone with the file can rebuild the same environment, on a different machine, months later.

Imagine someone in your lab hands you a new piece of code that you're going to use for analysis

In [ ]:
import optimus_prime

optimus_prime.defeat_decepticons()

That traceback is not a bug in the code. It's a missing `environment.yml`. (I discuss tracebacks a little later)

# Problem 2) Reproducibility

I am certain we all agree that it is important/essential that science and the scientific process are "reproducible" but what exactly does that mean?

Quick survivor game - everyone raise your hand. Lower it when you read a statement that is not 100% definitely true.

Can you re-run (all) your code and get the same answer as what is in your paper? Can someone else in your group do this? 

Can someone download all your software, run it, and get the same result?

Can someone read your paper, do a clean room implementation (or use another code), access the data source, and reproduce your results independantly?

Which one of these is "reproducibility"?

**Can you re-run your own calculation later and get the same result?**

This is easy (right?). 

Store everything in a repo with the aforementioned `environment.yml` and it should be straightforward.

This is not reproducibility, it's *repeatability*.

**Can someone download everything you used — code, data, environment specification — and run it independently to get the same result?**

This is a higher bar. It doesn't rely on your lab's shared cluster, your particular file paths, or you being available to answer questions.

This still isn't reproducibility, it's *replicability*. 

Not only must the software and data be accessible, there can be nothing hard coded *and* it needs to come with an instructions manual.

**Can someone read the *description* of your method in a paper and then implement it independently (possibly in a different language, with different code), obtain the same data, and reach the same scientific conclusion?**

Your code does not matter in this case. Have you described clearly and completely the method used so that it can be rebuilt from scratch?

This is *reproducibility*.

(and there's a reason no one's hand was still up at the end of the survivors game...)

Most projects stop at repeatability. 

The code runs, on your machine, for you. Throw it on the #arXiv, call it a day.

A seeded random number generator and a clean environment file get you to replicability. 

These are a good habit that do not require extra research effort.

Reproducibility is a property of the *paper*, not just the code — it depends on whether the methods section actually contains enough information to rebuild the analysis without access to your repository at all.

The most common enemies of reproducibility are random numbers and hard coding. 

#### Problem 2a) randomness

In [ ]:
import random

print(random.random())
print(random.random())

Run that cell again. Different numbers, every time, for anyone who runs this code — including you, next week.

Setting the seed for the random number generator produces the same sequence of (psuedo) random numbers. 

In [ ]:
import random

random.seed(42)
print(random.random())
print(random.random())

In [ ]:
# the equivalent pattern in numpy
import numpy as np

rng = np.random.default_rng(seed=42)
print(rng.random())
print(rng.random())

#### Problem 2b) hardcoded parameters

In [ ]:
# deep, deep inside Optimus Prime's source code
threshold = 0.037
learning_rate = 0.0012
data_path = "/Users/colin/Desktop/final_data_v3.csv"

Are you confident that you will know why `threshold` was set to 0.037 six months from now?

Throw any and all parameters to run the code into a config dictionary (or a config file that gets read in):

In [ ]:
CONFIG = {
    "threshold": 0.037,
    "learning_rate": 0.0012,
    "data_path": "data/raw/measurements.csv",
}

This also makes the parameters of the analysis part of your git history. If a value changes, there's a commit that says when and, ideally, why.

Seeded random numbers and externalized parameters move a project from repeatable to replicable. 

And the best part is... *this is not that hard*. Do it from the beginning because you aren't going to remember how/why you made these choices six months later. 

## Problem 3) Debugging

A traceback is Python telling you, in order, exactly how the software got to the line that failed.

The first time you get one it's scary. 

(like, really scary, like why are there now hundreds of lines of red code on my screen - Claude help!!!!!!)



Tracebacks can be easy to misread, the key is to consider if from the bottom up.

In [ ]:
def load_measurements(path):
    with open(path) as f:
        return f.read()

def compute_average(values):
    return sum(values) / len(values)

def analyze(path):
    data = load_measurements(path)
    return compute_average(data)

analyze("optimus_prime.csv")

How do we interpret this?

- **Bottom line**: the error type and message. Start here.

- **Just above it**: the exact line that failed.

- **Further up**: the chain of function calls that led there, oldest call at the top.

- Lines pointing into library code you didn't write are usually context, not the cause.

I have a secret to share with you: 

adding 14 different `print()` statements in your code to figure out what went wrong is not actually "debugging."

I also have good news: `pdb` does the same thing, but faster.

In [ ]:
def compute_average(values):
    breakpoint()   # execution pauses here and drops into a debugger
    return sum(values) / len(values)

Here is what you need to know for `pdb`:

| Command | Operation |
|---|---|
| `n` | next line |
| `s` | step *into* a function call |
| `c` | continue running |
| `p var` | print a variable |
| `l` | list surrounding code |
| `q` | quit |

Here are four common bugs that frequently pop up in scientific software: 

#### Problem 3a) Indicies that are off-by-one

In [ ]:
items = ["a", "b", "c"]
for i in range(1, len(items)):   # skips items[0]
    print(items[i])

The most common indexing bug in any language with zero-based arrays.

#### Problem 3b) The mutable default argument

In [ ]:
def add_result(value, results=[]):
    results.append(value)
    return results

print(add_result(1))
print(add_result(2))

The list is created once, when the function is *defined*, not fresh on every call — so it keeps everything from every previous call.

In [ ]:
def add_result(value, results=None):
    if results is None:
        results = []
    results.append(value)
    return results

print(add_result(1))
print(add_result(2))

#### Problem 3c) Floating point comparisons

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

Binary floating point can't represent most decimal fractions exactly, the same way 1/3 has no clean ending in base 10.

In [ ]:
import math
print(math.isclose(0.1 + 0.2, 0.3))

#### Problem 3d) NaN

In [ ]:
import numpy as np

values = np.array([1.0, 2.0, np.nan, 4.0])
print(values.mean())      # nan — no error, no warning

This doesn't raise an exception. It just produces a wrong number that can propagate quietly into a figure or a table.

In [ ]:
print(np.nanmean(values))

## Problem 4) Profiling

Raise your hand if you like waiting!

If there's a problem with your code debugging helps you find it. 

Profiling helps you figure out how to get correct code to run faster. 

#### Problem 4a) How long does it take?

In [ ]:
import time

start = time.perf_counter()
total = sum(i**2 for i in range(1_000_000))
end = time.perf_counter()

print(f"Took {end - start:.4f} seconds")

`time.perf_counter()` is preferred over `time.time()` for this — it's a clock built specifically for measuring short durations, and isn't affected by the system clock being adjusted.

#### Problem 4b) No seriously, how long does it take? `%timeit`

Timing manually has a problem: a single run is noisy. Background processes, cache effects, and garbage collection can all shift a single measurement.

In [ ]:
%timeit sum(i**2 for i in range(1_000_000))

`%timeit` is an IPython magic command — it re-runs the line many times and reports statistics instead of a single number.

How do we understand output that looks like this? 

`31.2 ms ± 412 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)`

- **7 runs**: the whole measurement is repeated 7 times, to see how much the timing itself varies

- **10 loops each**: within each run, the line is executed 10 times in a row and averaged, since a single execution can be too fast to time precisely

- `IPython` picks both numbers automatically, increasing the loop count for very fast code so the measurement isn't dominated by timer overhead

#### Problem 4c) Stop playing, I need to know how long it take! `%%timeit`

In [ ]:
%%timeit
values = []
for i in range(100_000):
    values.append(i**2)
total = sum(values)

`%timeit` times one line. `%%timeit`, as the first line of a cell, times everything in that cell.

Now that `%timeit` has told us something is slow, what do we do?

Use a profiler to figure out *which* part is slow. The standard tool for `python` is `cProfile`. 

In [ ]:
import cProfile

def load_data(n):
    return [i for i in range(n)]

def clean_data(data):
    # deliberately inefficient: rebuilds the list every iteration
    cleaned = []
    for x in data:
        cleaned = cleaned + [x * 2]
    return cleaned

def summarize(data):
    return sum(data) / len(data)

def run_pipeline(n):
    data = load_data(n)
    data = clean_data(data)
    return summarize(data)

cProfile.run("run_pipeline(5000)")

The output will look something like this:

```
         5010 function calls in 0.842 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     5000    0.831    0.000    0.831    0.000  clean_data
        1    0.009    0.009    0.840    0.840  run_pipeline
        1    0.001    0.001    0.001    0.001  load_data
        1    0.000    0.000    0.000    0.000  summarize
```

- **ncalls**: how many times the function was called

- **tottime**: time spent in that function alone, excluding functions it called

- **cumtime**: time spent in that function *and* everything it called

Here, almost all the time is in `clean_data`, not because it does more work, but because rebuilding the list on every iteration is quadratic instead of linear.

It is **hard** to guess which part of a code is running slow. 

Profile before rewriting anything! Profile before parallelizing. Profile to actually learn how to speed things up. 

## Save yourself from yourself

Isolate your environment. 

Write down how you got your answer.

Read the last line of the traceback before you panic.

Measure before you optimize.